# Industry Opportunity Component

### 0. Overview

This notebook constructs the industry opportunity component of the Opportunity Layer. The component uses the industry attribute retained in each representative demographic profile to refine the destination category of commuting trips.

Where an industry category has a clear semantic correspondence with a POI category, the general Employment destination is replaced by the corresponding category. Industries incluing in the Employment POI retain Employment as the final destination.

Objectives: 

1. Load and inspect the industry categories retained in the demographic layer.
2. Load the harmonised POI ontology.
3. Construct the Industry-to-POI mapping.
4. Validate category coverage and mapping targets.
5. Export the validated mapping for final framework integration.

### 1. Import Libraries

In [1]:
from pathlib import Path
import json

import pandas as pd

### 2. Set Up File Paths

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"

DEMOGRAPHIC_PROFILE_PATH = (
    OUTPUT_DIR
    / "demographic_profiles"
    / "constrained_representative_population_space.parquet"
)

INDUSTRY_OUTPUT_DIR = (
    OUTPUT_DIR
    / "opportunity_layer"
)

INDUSTRY_OUTPUT_PATH = (
    INDUSTRY_OUTPUT_DIR
    / "industry_to_poi_mapping.json"
)

### 4. Load Industry Categories

The industry attribute is loaded from the constrained representative demographic profiles. Only the Industry column is required because this component defines destination refinements at the industry-category level rather than at the individual-profile level.

In [3]:
industry_data = pd.read_parquet(
    DEMOGRAPHIC_PROFILE_PATH,
    columns=["Industry"]
)

industry_data.head()

,Industry
0,Does not apply
1,Does not apply
2,Does not apply
3,Does not apply
4,Does not apply


## 5. Data Inspection

In [4]:
print(f"Rows loaded: {len(industry_data):,}")
print(
    f"Missing Industry values: "
    f"{industry_data['Industry'].isna().sum():,}"
)
print(
    f"Unique Industry categories: "
    f"{industry_data['Industry'].nunique()}"
)

Rows loaded: 2,386,560
Missing Industry values: 0
Unique Industry categories: 19


In [5]:
industry_categories = sorted(
    industry_data["Industry"].unique().tolist())

display(industry_categories)

['A Agriculture, forestry and fishing',
 'B Mining and quarrying',
 'C Manufacturing',
 'D Electricity, gas, steam and air conditioning supply',
 'Does not apply',
 'E Water supply; sewerage, waste management and remediation activities',
 'F Construction',
 'G Wholesale and retail trade; repair of motor vehicles and motor cycles',
 'H Transport and storage',
 'I Accommodation and food service activities',
 'J Information and communication',
 'K Financial and insurance activities',
 'L Real estate activities',
 'M Professional, scientific and technical activities',
 'N Administrative and support service activities',
 'O Public administration and defence; compulsory social security',
 'P Education',
 'Q Human health and social work activities',
 'R, S, T, U Other']

In [6]:
pd.DataFrame(industry_categories, columns=["Industry"])

,Industry
0,"A Agriculture, forestry and fishing"
1,B Mining and quarrying
2,C Manufacturing
3,"D Electricity, gas, steam and air conditioning..."
4,Does not apply
5,"E Water supply; sewerage, waste management and..."
6,F Construction
7,G Wholesale and retail trade; repair of motor ...
8,H Transport and storage
9,I Accommodation and food service activities


### 6. Industry to POI Category Mapping

A mapping was created between the 19 industry categories retained in the demographic layer and the framework POI categories. Industries with a clear semantic correspondence were assigned a specific POI category, while broader or ambiguous industries were mapped to Employment. Does not apply was assigned None because it does not trigger industry-specific refinement.

In [7]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [8]:
from ontology.INDUSTRY_TO_POI_MAPPING import INDUSTRY_TO_POI_MAPPING

In [9]:
unmapped_industries = (
    set(industry_categories)
    - set(INDUSTRY_TO_POI_MAPPING)
)

unexpected_industries = (
    set(INDUSTRY_TO_POI_MAPPING)
    - set(industry_categories)
)

print(f"Unmapped industries: {unmapped_industries}")
print(f"Unexpected industries: {unexpected_industries}")

Unmapped industries: set()
Unexpected industries: set()
